# SGBR C-01 and E1 on Kaggle CPU

1. Import this notebook into a **private** Kaggle Notebook. Set **Accelerator → None** and enable **Internet**.
2. Add a Kaggle Secret named `GITHUB_TOKEN` and grant this notebook access. Use a fine-grained token with **Contents: Read** for `AIVIETNAM-AIO-DucTam/SGBR-Reproduction`.
3. Select **Run All**. To retain notebook outputs, use **Save Version → Run All**. The configuration defaults to `main`. When testing an unmerged implementation, choose its branch or full commit SHA.
4. Watch the stage messages and the periodic E1 percentage, Java RSS and cgroup memory readings. Download `/kaggle/working/results/` after the run.

C-01 builds or verifies and reuses the pinned MOA jar, then runs the eight-row smoke test. E1 receives that jar and its computed SHA-256 automatically. It runs the full NZEnergy stream with the authors' Sgb(Arf) configuration, seed 1, and a 16 GiB maximum heap. Completion requires the full 3,100,666 rows in the evaluator CSV and a successful Java exit.

Leave enough time for E1 and exports. A prior Kaggle attempt was still running after roughly three hours. Set a timeout only if you want a controlled stop before your session ends. The progress percentage comes from MOA and any associated row count is an estimate. If E1 stops, the logs and last monitoring data are retained, subject to Kaggle preserving the session's files. E1 must restart from the beginning because this runner does not save learner state.

In [ ]:
from pathlib import Path
import time

STARTED = time.monotonic()
GITHUB_REF = "main"
GITHUB_REFRESH = True  # Refresh the selected branch. Exact SHAs can be reused.
TIMEOUT_SECONDS = None  # No runner timeout by default.
PROGRESS_INTERVAL_SECONDS = 60
SESSION_LIMIT_HOURS = None  # Optional, from the active Kaggle session.
SESSION_LIMIT_SOURCE = None


In [ ]:
import os
import re
import subprocess
from tempfile import TemporaryDirectory

PROJECT_URL = "https://github.com/AIVIETNAM-AIO-DucTam/SGBR-Reproduction.git"
DEFAULT_CHECKOUT = Path("/kaggle/working/SGBR-Reproduction")


def project_git(folder, *args, env=None):
    try:
        return subprocess.run(
            ["git", "-C", str(folder), *args], capture_output=True, text=True,
            env=env, timeout=180, check=False,
        )
    except subprocess.TimeoutExpired:
        raise RuntimeError("Git timed out. Enable Internet in Kaggle and check GitHub connectivity.") from None
    except OSError:
        raise RuntimeError("Could not start Git. Check that Git is available in the notebook.") from None


def project_commit(folder):
    if not (folder / ".git").exists():
        return None
    head = project_git(folder, "rev-parse", "--verify", "HEAD^{commit}")
    if head.returncode or not re.fullmatch(r"[0-9a-f]{40}", head.stdout.strip()):
        raise RuntimeError("Could not resolve the project checkout's Git commit.")
    return head.stdout.strip()


def fetch_project(folder, revision):
    try:
        from kaggle_secrets import UserSecretsClient

        token = UserSecretsClient().get_secret("GITHUB_TOKEN")
    except Exception:
        raise RuntimeError(
            "Cannot read Kaggle Secret GITHUB_TOKEN. Add the fine-grained token in "
            "Kaggle Secrets and grant this notebook access to it."
        ) from None
    if not isinstance(token, str) or not token.strip():
        raise RuntimeError("Kaggle Secret GITHUB_TOKEN is empty. Set it in Kaggle Secrets.")

    with TemporaryDirectory(prefix="sgbr-git-auth-") as temporary:
        askpass = Path(temporary) / "askpass"
        askpass.write_text(
            '#!/bin/sh\ncase "$1" in\n'
            '  "Username for \'https://github.com\': "*) printf "%s\\n" x-access-token ;;\n'
            '  "Password for \'https://x-access-token@github.com\': "*) '
            'printf "%s\\n" "$SGBR_GIT_SECRET" ;;\n'
            '  *) exit 1 ;;\nesac\n',
            encoding="utf-8",
        )
        askpass.chmod(0o700)
        env = {
            key: value for key, value in os.environ.items()
            if not key.startswith("GIT_") and key != "SSLKEYLOGFILE"
        }
        env.update(
            LC_ALL="C", GIT_ASKPASS=str(askpass), GIT_TERMINAL_PROMPT="0",
            GIT_CONFIG_NOSYSTEM="1", GIT_CONFIG_GLOBAL=os.devnull, SGBR_GIT_SECRET=token,
        )
        fetch = project_git(
            folder, "-c", "credential.helper=", "-c", "http.followRedirects=false",
            "-c", "http.sslVerify=true", "fetch", "--quiet", "--depth=1", "--no-tags",
            "--", PROJECT_URL, revision, env=env,
        )
    if not fetch.returncode:
        return

    # Classify the captured diagnostic without replaying Git or server output.
    diagnostic = fetch.stderr.lower()
    if any(message in diagnostic for message in (
        "could not resolve", "failed to connect", "couldn't connect", "network is unreachable",
        "connection timed out", "connect tunnel failed", "proxy", "connection reset",
    )):
        raise RuntimeError("Could not reach GitHub. Enable Internet in Kaggle notebook settings and retry.")
    if any(message in diagnostic for message in (
        "authentication failed", "repository not found", "401", "403",
        "access denied", "could not read username", "could not read password", "invalid username or token",
    )):
        raise RuntimeError(
            "GitHub authentication or repository access failed. Check that GITHUB_TOKEN is valid "
            "and has Contents: Read access to AIVIETNAM-AIO-DucTam/SGBR-Reproduction."
        )
    if any(message in diagnostic for message in ("couldn't find remote ref", "not our ref", "unadvertised object")):
        raise RuntimeError("GitHub could not fetch GITHUB_REF. Check the branch name or full commit SHA.")
    raise RuntimeError("GitHub fetch failed. Verify Internet, token permissions, and GITHUB_REF, then retry.")


def prepare_project_checkout(target, ref, refresh=False):
    if not target.parent.is_dir():
        raise RuntimeError("Automatic checkout requires /kaggle/working. Run in Kaggle or set CHECKOUT.")
    if not isinstance(ref, str) or not ref or ref.startswith("-"):
        raise ValueError("Set GITHUB_REF to a branch name or full 40-character commit SHA.")
    is_sha = re.fullmatch(r"[0-9a-fA-F]{40}", ref) is not None
    revision = ref.lower() if is_sha else f"refs/heads/{ref}"
    if not is_sha and project_git(target.parent, "check-ref-format", revision).returncode:
        raise ValueError("Set GITHUB_REF to a valid branch name or full 40-character commit SHA.")

    if target.exists() or target.is_symlink():
        if target.is_symlink() or not (target / ".git").is_dir():
            raise RuntimeError("Automatic checkout path is occupied. Set CHECKOUT to use it explicitly.")
        origin = project_git(target, "config", "--local", "--get", "remote.origin.url")
        if origin.returncode or origin.stdout.strip() != PROJECT_URL:
            raise RuntimeError("Automatic checkout path has a different Git origin. Set CHECKOUT to use it explicitly.")
        status = project_git(target, "status", "--porcelain")
        if status.returncode or status.stdout.strip():
            raise RuntimeError("Automatic checkout has local changes. Preserve them or select an explicit CHECKOUT.")
        head = project_commit(target)
        cached_ref = f"refs/remotes/origin/{ref}"
        cached = head == revision if is_sha else (
            project_git(target, "rev-parse", "--verify", cached_ref).stdout.strip() == head
        )
        if cached and not refresh:
            return head
        return checkout_project_revision(target, ref, revision, is_sha)

    # Build a shallow checkout in a sibling directory and publish it only on success.
    with TemporaryDirectory(prefix=".sgbr-checkout-", dir=target.parent) as temporary:
        folder = Path(temporary) / "repo"
        folder.mkdir()
        initial = project_git(folder, "init", "--quiet")
        if initial.returncode or project_git(folder, "remote", "add", "origin", PROJECT_URL).returncode:
            raise RuntimeError("Could not initialize the private checkout in /kaggle/working.")
        head = checkout_project_revision(folder, ref, revision, is_sha)
        folder.rename(target)
        return head


def checkout_project_revision(folder, ref, revision, is_sha):
    fetch_project(folder, revision)
    fetched = project_git(folder, "rev-parse", "--verify", "FETCH_HEAD^{commit}")
    sha = fetched.stdout.strip()
    if fetched.returncode or not re.fullmatch(r"[0-9a-f]{40}", sha) or (is_sha and sha != revision):
        raise RuntimeError("GitHub did not return the requested project commit.")
    if project_git(folder, "-c", "core.hooksPath=/dev/null", "checkout", "--quiet", "--detach", sha).returncode:
        raise RuntimeError("Could not check out the fetched project commit.")
    if not is_sha and project_git(folder, "update-ref", f"refs/remotes/origin/{ref}", sha).returncode:
        raise RuntimeError("Could not record the fetched project branch.")
    return project_commit(folder)


In [ ]:
import datetime
import sys

if not DEFAULT_CHECKOUT.parent.is_dir():
    raise RuntimeError("Open this notebook in a Kaggle CPU session with /kaggle/working available.")
stamp = datetime.datetime.now(datetime.timezone.utc).isoformat(timespec="seconds")
print(f"[{stamp}] Resolving the private project checkout", flush=True)
refresh = GITHUB_REFRESH and re.fullmatch(r"[0-9a-fA-F]{40}", GITHUB_REF) is None
commit = prepare_project_checkout(DEFAULT_CHECKOUT, GITHUB_REF, refresh)
scripts = DEFAULT_CHECKOUT / "scripts"
modules = ("sgbr_c01_e1_integrated", "e1_memory_feasibility", "kaggle_environment_smoke")
for name in modules:
    if not (scripts / f"{name}.py").is_file():
        raise FileNotFoundError(f"{scripts / f'{name}.py'} is missing. Select a ref containing the integrated pipeline.")
print(f"Project commit: {commit}", flush=True)

# Import fresh modules from the verified checkout and keep it free of bytecode files.
save_bytecode = sys.dont_write_bytecode
sys.dont_write_bytecode = True
sys.path.insert(0, str(scripts))
try:
    for name in modules:
        sys.modules.pop(name, None)
    from sgbr_c01_e1_integrated import run_pipeline
finally:
    sys.path.remove(str(scripts))
    sys.dont_write_bytecode = save_bytecode

outcome = run_pipeline(
    DEFAULT_CHECKOUT.parent, commit, timeout_seconds=TIMEOUT_SECONDS,
    progress_interval=PROGRESS_INTERVAL_SECONDS, session_limit_hours=SESSION_LIMIT_HOURS,
    session_limit_source=SESSION_LIMIT_SOURCE, started=STARTED,
)
if outcome:
    raise RuntimeError("E1 did not complete. Review results/manifest/e1_memory_feasibility.json and its logs.")
